In [ ]:
# 📦 Import Libraries
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import classification_report
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Start timer
start_time = time.time()
print("🔍 Starting WiDS 2025 Enhanced Classification Pipeline...")

# 📂 Path setup
data_dir = r"C:\Users\sanja\Desktop\WiDS"
train_dir = os.path.join(data_dir, "TRAIN_NEW")
test_dir = os.path.join(data_dir, "TEST")
output_dir = os.path.join(data_dir, "submissions")
os.makedirs(output_dir, exist_ok=True)

# 🔄 Define constants and target variables
ID_COL = 'participant_id'
TARGETS = ['Sex_F', 'ADHD_Outcome']
PCA_COMPONENTS = 175
VARIANCE_THRESHOLD = 0.0005

# 📥 Load TRAIN data
print("\n📂 Loading TRAIN datasets...")
train_conn = pd.read_csv(os.path.join(train_dir, "TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv"))
train_quant = pd.read_excel(os.path.join(train_dir, "TRAIN_QUANTITATIVE_METADATA_new.xlsx"))
train_cat = pd.read_excel(os.path.join(train_dir, "TRAIN_CATEGORICAL_METADATA_new.xlsx"))
train_labels = pd.read_excel(os.path.join(train_dir, "TRAINING_SOLUTIONS.xlsx"))

# Verify participant_id is present in all dataframes
for df, name in [(train_conn, "Connectome"), (train_quant, "Quantitative"), 
                 (train_cat, "Categorical"), (train_labels, "Labels")]:
    if ID_COL not in df.columns:
        raise ValueError(f"{ID_COL} not found in {name} dataframe!")
    print(f"{name} shape: {df.shape}, Has {ID_COL}: {ID_COL in df.columns}")

# 📥 Load TEST data
print("\n📂 Loading TEST datasets...")
test_conn = pd.read_csv(os.path.join(test_dir, "TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv"))
test_quant = pd.read_excel(os.path.join(test_dir, "TEST_QUANTITATIVE_METADATA.xlsx"))
test_cat = pd.read_excel(os.path.join(test_dir, "TEST_CATEGORICAL.xlsx"))

# Verify participant_id is present in all test dataframes
for df, name in [(test_conn, "Test Connectome"), (test_quant, "Test Quantitative"), 
                 (test_cat, "Test Categorical")]:
    if ID_COL not in df.columns:
        raise ValueError(f"{ID_COL} not found in {name} dataframe!")
    print(f"{name} shape: {df.shape}, Has {ID_COL}: {ID_COL in df.columns}")

# 🔄 Get column categories
connectome_cols = [col for col in train_conn.columns if col != ID_COL]
quant_cols = [col for col in train_quant.columns if col != ID_COL]
cat_cols = [col for col in train_cat.columns if col != ID_COL]

# 🧹 Merge train datasets - ENSURING patient_id remains first column
print("\n🔄 Merging TRAIN datasets on participant_id...")
# First merge quantitative and categorical
train_meta = pd.merge(train_quant, train_cat, on=ID_COL, how='outer')
# Then merge with connectome and labels
train_full = pd.merge(train_conn, train_meta, on=ID_COL, how='inner')
train_full = pd.merge(train_full, train_labels, on=ID_COL, how='inner')

# Ensure patient_id is the first column
cols = [ID_COL] + [col for col in train_full.columns if col != ID_COL]
train_full = train_full[cols]

print(f"Final train dataset shape: {train_full.shape}")
print(f"First column is {train_full.columns[0]}")

# Verify we didn't lose any patients during the merge
print(f"Original train connectome patients: {train_conn[ID_COL].nunique()}")
print(f"Final merged train patients: {train_full[ID_COL].nunique()}")

# 🧹 Merge test datasets - ENSURING patient_id remains first column
print("\n🔄 Merging TEST datasets on participant_id...")
# First merge quantitative and categorical
test_meta = pd.merge(test_quant, test_cat, on=ID_COL, how='outer')
# Then merge with connectome
test_full = pd.merge(test_conn, test_meta, on=ID_COL, how='inner')

# Ensure patient_id is the first column
cols = [ID_COL] + [col for col in test_full.columns if col != ID_COL]
test_full = test_full[cols]

print(f"Final test dataset shape: {test_full.shape}")
print(f"First column is {test_full.columns[0]}")

# Verify we didn't lose any patients during the merge
print(f"Original test connectome patients: {test_conn[ID_COL].nunique()}")
print(f"Final merged test patients: {test_full[ID_COL].nunique()}")

# 🧮 Separate features and targets
X = train_full.drop(columns=TARGETS)  # Keep patient_id
y = train_full[TARGETS]
X_test = test_full  # Keep patient_id in X_test

# 📊 Display class distributions
print("\n📊 Class distribution for Sex_F:")
print(y['Sex_F'].value_counts())

print("\n📊 Class distribution for ADHD_Outcome:")
print(y['ADHD_Outcome'].value_counts())

# 🧹 Extract patient_ids before preprocessing
train_ids = X[ID_COL].values
test_ids = X_test[ID_COL].values

# Now remove patient_id for preprocessing
X = X.drop(columns=[ID_COL])
X_test = X_test.drop(columns=[ID_COL])

# 🧹 Handle missing values and encode categorical features
print("\n🔄 Preprocessing data...")

# For numerical features
num_cols = X.select_dtypes(include=np.number).columns
for col in num_cols:
    if X[col].isna().sum() > 0:
        # Fill missing values based on distribution
        if abs(X[col].skew()) > 1:
            X[col] = X[col].fillna(X[col].median())
            X_test[col] = X_test[col].fillna(X[col].median())
        else:
            X[col] = X[col].fillna(X[col].mean())
            X_test[col] = X_test[col].fillna(X[col].mean())

# For categorical features
cat_features = X.select_dtypes(exclude=np.number).columns
encoders = {}

for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    encoders[col] = le

# Fill any remaining missing values
X = X.fillna(0)
X_test = X_test.fillna(0)

# 🔄 Apply dimensionality reduction to connectome data
print("\n🔄 Applying PCA to connectome data...")
conn_pca = PCA(n_components=PCA_COMPONENTS, random_state=42)
X_conn = X[connectome_cols]
X_test_conn = X_test[connectome_cols]

X_conn_pca = conn_pca.fit_transform(X_conn)
X_test_conn_pca = conn_pca.transform(X_test_conn)

print(f"Explained variance ratio sum: {np.sum(conn_pca.explained_variance_ratio_):.4f}")
print(f"Connectome data reduced from {len(connectome_cols)} to {PCA_COMPONENTS} features")

# 🔄 Apply variance threshold to metadata
print("\n🔄 Filtering metadata features...")
meta_cols = [col for col in X.columns if col not in connectome_cols]
X_meta = X[meta_cols]
X_test_meta = X_test[meta_cols]

vt = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_meta_vt = vt.fit_transform(X_meta)
X_test_meta_vt = vt.transform(X_test_meta)

selected_meta_cols = np.array(meta_cols)[vt.get_support()]
print(f"Metadata features reduced from {len(meta_cols)} to {len(selected_meta_cols)}")

# 🔄 Combine reduced features - DIRECT NUMPY APPROACH TO AVOID COLUMN NAME ISSUES
# Convert everything to numpy arrays and bypass the DataFrame column name issue
X_combined = np.hstack([X_conn_pca, X_meta_vt])
X_test_combined = np.hstack([X_test_conn_pca, X_test_meta_vt])

print(f"Combined train data shape: {X_combined.shape}")
print(f"Combined test data shape: {X_test_combined.shape}")

# 🔄 Scale features - USE NUMPY ARRAYS DIRECTLY TO AVOID STANDARDSCALER ERROR
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)
X_test_scaled = scaler.transform(X_test_combined)

# 📈 Create cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🔄 Define model training and evaluation function
def train_evaluate_model(X, y, X_test, target, cv=cv):
    print(f"\n🧬 PART {1 if target == 'Sex_F' else 2}: {target.upper()} PREDICTION")
    
    # Apply appropriate resampling based on target
    if target == 'Sex_F':
        print("✅ Applying SMOTE for Sex_F prediction...")
        resampler = SMOTE(random_state=42)
        X_resampled, y_resampled = resampler.fit_resample(X, y)
        print(f"✅ After SMOTE: {np.bincount(y_resampled)}")
        
        # Create model for Sex_F
        catboost = CatBoostClassifier(
            verbose=0, random_state=42, depth=7, learning_rate=0.02, 
            iterations=1000, l2_leaf_reg=5, early_stopping_rounds=100
        )
        
        lgbm = lgb.LGBMClassifier(
            n_estimators=800, learning_rate=0.02, num_leaves=31, 
            max_depth=7, random_state=42
        )
        
        model = VotingClassifier(
            estimators=[
                ('catboost', catboost),
                ('lgbm', lgbm)
            ],
            voting='soft'
        )
    else:  # ADHD_Outcome
        print("✅ Applying SMOTETomek for ADHD prediction...")
        resampler = SMOTETomek(random_state=42)
        X_resampled, y_resampled = resampler.fit_resample(X, y)
        print(f"✅ Original class distribution: {np.bincount(y)}")
        print(f"✅ After SMOTETomek: {np.bincount(y_resampled)}")
        
        # Calculate class weights
        class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
        class_weights_dict = dict(zip(np.unique(y), class_weights))
        print(f"✅ Class weights: {class_weights_dict}")
        
        # Create models for ADHD
        catboost = CatBoostClassifier(
            verbose=0, random_state=42, depth=8, learning_rate=0.02, 
            iterations=2000, l2_leaf_reg=3, class_weights=class_weights_dict
        )
        
        lgbm = lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.02, num_leaves=31,
            max_depth=8, class_weight=class_weights_dict, random_state=42
        )
        
        xgb_model = xgb.XGBClassifier(
            n_estimators=1000, learning_rate=0.02, max_depth=8,
            scale_pos_weight=class_weights_dict[0]/class_weights_dict[1], 
            random_state=42
        )
        
        model = StackingClassifier(
            estimators=[
                ('catboost', catboost),
                ('lgbm', lgbm),
                ('xgb', xgb_model)
            ],
            final_estimator=LogisticRegression(class_weight='balanced'),
            cv=3
        )
    
    # Evaluate with cross-validation
    print(f"🔄 Evaluating {target} model with cross-validation...")
    cv_preds = cross_val_predict(model, X_resampled, y_resampled, cv=cv)
    
    print(f"\n==== Cross-Validation Report for {target} ====")
    print(classification_report(y_resampled, cv_preds))
    
    # Train final model
    print(f"🔄 Training final {target} model on complete dataset...")
    model.fit(X_resampled, y_resampled)
    
    # Generate predictions
    test_preds = model.predict(X_test)
    
    # If possible, get prediction probabilities
    if hasattr(model, 'predict_proba'):
        test_probs = model.predict_proba(X_test)[:, 1]
        return test_preds, test_probs, model
    else:
        return test_preds, None, model

# 🧪 Train and evaluate models for each target
sex_preds, sex_probs, sex_model = train_evaluate_model(X_scaled, y['Sex_F'].values, X_test_scaled, 'Sex_F')
adhd_preds, adhd_probs, adhd_model = train_evaluate_model(X_scaled, y['ADHD_Outcome'].values, X_test_scaled, 'ADHD_Outcome')

# 🔄 Post-processing for optimizing female ADHD cases (which get 2x weight)
print("\n🧬 PART 3: OPTIMIZING FOR FEMALE ADHD CASES (2x WEIGHT)")

# Check if we have probabilities for both models
if sex_probs is not None and adhd_probs is not None:
    # Identify borderline cases
    borderline_female = (sex_probs > 0.45) & (sex_probs < 0.55)
    borderline_adhd = (adhd_probs > 0.45) & (adhd_probs < 0.55)
    
    # Get final predictions with optimization
    final_sex_preds = sex_preds.copy()
    final_adhd_preds = adhd_preds.copy()
    
    # Special handling for high-confidence female ADHD cases
    joint_prob = sex_probs * adhd_probs
    high_joint_prob = joint_prob > 0.5
    
    # Borderline case adjustments to optimize for female ADHD cases
    final_sex_preds[borderline_female & (adhd_probs > 0.7)] = 1
    final_adhd_preds[borderline_adhd & (sex_probs > 0.7)] = 1
else:
    final_sex_preds = sex_preds
    final_adhd_preds = adhd_preds

# 📊 Analyze predictions
print("\n📋 Distribution Analysis of Final Predictions:")

# Check distribution of predictions
print("\n==== Test Sex_F Predictions Distribution ====")
sex_counts = np.bincount(final_sex_preds)
print(f"Sex_F=0: {sex_counts[0]}, Sex_F=1: {sex_counts[1]}")

print("\n==== Test ADHD_Outcome Predictions Distribution ====")
adhd_counts = np.bincount(final_adhd_preds)
print(f"ADHD_Outcome=0: {adhd_counts[0]}, ADHD_Outcome=1: {adhd_counts[1]}")

# Check female ADHD cases specifically
female_adhd_count = np.sum((final_sex_preds == 1) & (final_adhd_preds == 1))
print(f"\n==== Female ADHD Cases (2x weight): {female_adhd_count} ====")

# 📋 Reporting on First 155 Test Rows
print("\n📋 Reporting on First 155 Test Rows:")
pred_test_sex_first155 = final_sex_preds[:155]
pred_test_adhd_first155 = final_adhd_preds[:155]

print("\n==== FIRST 155 Test Sex_F Predictions Distribution ====")
print(np.unique(pred_test_sex_first155, return_counts=True))

print("\n==== FIRST 155 Test ADHD_Outcome Predictions Distribution ====")
print(np.unique(pred_test_adhd_first155, return_counts=True))

# 📥 Create and save final submission
# IMPORTANT: Use the test_ids we extracted earlier to ensure correct patient_id alignment
submission = pd.DataFrame({
    ID_COL: test_ids,  # This is the correct test_ids extracted before preprocessing
    'Sex_F': final_sex_preds,
    'ADHD_Outcome': final_adhd_preds
})

# Double-check ID_COL is the first column
if submission.columns[0] != ID_COL:
    submission = submission[[ID_COL] + [col for col in submission.columns if col != ID_COL]]

# Validation checks before saving
print("\n📋 Final Submission Format Check:")
print(f"First column is {submission.columns[0]}")
print(f"Submission shape: {submission.shape}")
print(f"First 5 rows of submission:")
print(submission.head())

# Save submission file
submission_path = os.path.join(output_dir, "submission_final_enhanced.csv")
submission.to_csv(submission_path, index=False)
print(f"\n✅ Enhanced submission saved to '{submission_path}'")

# Print execution time
total_time = time.time() - start_time
print(f"\n⏱️ Total execution time: {total_time/60:.2f} minutes")


🔍 Starting WiDS 2025 Enhanced Classification Pipeline...

📂 Loading TRAIN datasets...
Connectome shape: (1213, 19901), Has participant_id: True
Quantitative shape: (1213, 19), Has participant_id: True
Categorical shape: (1213, 10), Has participant_id: True
Labels shape: (1213, 3), Has participant_id: True

📂 Loading TEST datasets...
Test Connectome shape: (304, 19901), Has participant_id: True
Test Quantitative shape: (304, 19), Has participant_id: True
Test Categorical shape: (304, 10), Has participant_id: True

🔄 Merging TRAIN datasets on participant_id...
Final train dataset shape: (1213, 19930)
First column is participant_id
Original train connectome patients: 1213
Final merged train patients: 1213

🔄 Merging TEST datasets on participant_id...
Final test dataset shape: (304, 19928)
First column is participant_id
Original test connectome patients: 304
Final merged test patients: 304

📊 Class distribution for Sex_F:
Sex_F
0    797
1    416
Name: count, dtype: int64

📊 Class distribut

  File "C:\Users\sanja\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\sanja\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sanja\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sanja\anaconda3\Lib\subprocess.

[LightGBM] [Info] Number of positive: 638, number of negative: 637
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005637 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48891
[LightGBM] [Info] Number of data points in the train set: 1275, number of used features: 202
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500392 -> initscore=0.001569
[LightGBM] [Info] Start training from score 0.001569
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

In [1]:
# 📦 Import Libraries
import pandas as pd
import numpy as np
import os
import time
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.feature_selection import mutual_info_classif, SelectFromModel
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier, Pool
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Start timer
start_time = time.time()
print("🔍 Starting WiDS 2025 Ultra-Optimized Classification Pipeline...")

# 📂 Path setup
data_dir = r"C:\Users\sanja\Desktop\WiDS"
train_dir = os.path.join(data_dir, "TRAIN_NEW")
test_dir = os.path.join(data_dir, "TEST")
output_dir = os.path.join(data_dir, "submissions")
os.makedirs(output_dir, exist_ok=True)

# 🔄 Define constants and target variables
ID_COL = 'participant_id'
TARGETS = ['Sex_F', 'ADHD_Outcome']

# 📥 Load TRAIN data
print("\n📂 Loading TRAIN datasets...")
train_conn = pd.read_csv(os.path.join(train_dir, "TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv"))
train_quant = pd.read_excel(os.path.join(train_dir, "TRAIN_QUANTITATIVE_METADATA_new.xlsx"))
train_cat = pd.read_excel(os.path.join(train_dir, "TRAIN_CATEGORICAL_METADATA_new.xlsx"))
train_labels = pd.read_excel(os.path.join(train_dir, "TRAINING_SOLUTIONS.xlsx"))

# 📥 Load TEST data
print("\n📂 Loading TEST datasets...")
test_conn = pd.read_csv(os.path.join(test_dir, "TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv"))
test_quant = pd.read_excel(os.path.join(test_dir, "TEST_QUANTITATIVE_METADATA.xlsx"))
test_cat = pd.read_excel(os.path.join(test_dir, "TEST_CATEGORICAL.xlsx"))

# 🔄 Get column categories
connectome_cols = [col for col in train_conn.columns if col != ID_COL]
quant_cols = [col for col in train_quant.columns if col != ID_COL]
cat_cols = [col for col in train_cat.columns if col != ID_COL]

# 🧹 Merge train datasets
print("\n🔄 Merging datasets...")
# First merge quantitative and categorical
train_meta = pd.merge(train_quant, train_cat, on=ID_COL, how='outer')
test_meta = pd.merge(test_quant, test_cat, on=ID_COL, how='outer')

# Then merge with connectome and labels
train_full = pd.merge(train_conn, train_meta, on=ID_COL, how='inner')
train_full = pd.merge(train_full, train_labels, on=ID_COL, how='inner')
test_full = pd.merge(test_conn, test_meta, on=ID_COL, how='inner')

print(f"Final train dataset shape: {train_full.shape}")
print(f"Final test dataset shape: {test_full.shape}")

# 🧮 Separate features and targets
X = train_full.drop(columns=TARGETS)
y = train_full[TARGETS]
X_test = test_full

# 🧹 Extract patient_ids before preprocessing
train_ids = X[ID_COL].values
test_ids = X_test[ID_COL].values

# Remove patient_id for preprocessing
X = X.drop(columns=[ID_COL])
X_test = X_test.drop(columns=[ID_COL])

# 🆕 CRITICAL ENHANCEMENT: Feature preprocessing and engineering
print("\n🔄 Advanced feature engineering...")

# 1. Preprocess all features first
# Handle missing values
X = X.fillna(X.median())
X_test = X_test.fillna(X.median())

# Encode categorical features
cat_features = X.select_dtypes(exclude=np.number).columns
for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# 2. Extract connectome-specific features for Sex_F
print("\n🔍 Extracting specialized connectome features for Sex_F...")
# Calculate top graph-based metrics from connectome
X_conn = X[connectome_cols].copy()
X_test_conn = X_test[connectome_cols].copy()

# Calculate network statistics
def extract_connectome_features(X_conn):
    # Node strength (sum of connections)
    strength = X_conn.sum(axis=1).values.reshape(-1, 1)
    
    # Mean positive and negative correlations
    pos_conn = X_conn.copy()
    pos_conn[pos_conn < 0] = 0
    pos_mean = pos_conn.mean(axis=1).values.reshape(-1, 1)
    
    neg_conn = X_conn.copy()
    neg_conn[neg_conn > 0] = 0
    neg_mean = neg_conn.mean(axis=1).values.reshape(-1, 1)
    
    # Variance in connections
    variance = X_conn.var(axis=1).values.reshape(-1, 1)
    
    # Quartile statistics
    q25 = X_conn.quantile(0.25, axis=1).values.reshape(-1, 1)
    q75 = X_conn.quantile(0.75, axis=1).values.reshape(-1, 1)
    iqr = (q75 - q25)
    
    # Row-wise statistics
    row_max = X_conn.max(axis=1).values.reshape(-1, 1)
    row_min = X_conn.min(axis=1).values.reshape(-1, 1)
    row_range = row_max - row_min
    
    # Calculate asymmetry (skewness-like measure)
    skew_like = X_conn.skew(axis=1).values.reshape(-1, 1)
    kurtosis_like = X_conn.kurtosis(axis=1).values.reshape(-1, 1)
    
    # Calculate absolute sum (total connection strength regardless of direction)
    abs_sum = np.abs(X_conn).sum(axis=1).values.reshape(-1, 1)
    
    # Top 10 highest absolute values per patient (flatten-sort-take top N approach)
    def get_top_n_abs(row, n=10):
        return np.sort(np.abs(row))[-n:]
    
    top_vals = np.array([get_top_n_abs(row) for row in X_conn.values])
    
    # Combine all features
    connectome_features = np.hstack([
        strength, pos_mean, neg_mean, variance, 
        q25, q75, iqr, row_max, row_min, row_range,
        skew_like, kurtosis_like, abs_sum, top_vals
    ])
    
    return connectome_features

# Extract these features (no need for parallelization for speed if issues)
train_connectome_features = extract_connectome_features(X_conn)
test_connectome_features = extract_connectome_features(X_test_conn)
print(f"Generated {train_connectome_features.shape[1]} specialized connectome features")

# 3. Find top 20 individual connectome columns for Sex_F
# Calculate mutual information between features and Sex_F target
print("\n🔍 Finding top connectome columns for Sex_F prediction...")
mi_scores = mutual_info_classif(X_conn, y['Sex_F'], random_state=42)
top_20_conn_indices = np.argsort(mi_scores)[-20:]
top_conn_cols = [connectome_cols[i] for i in top_20_conn_indices]
print(f"Top 20 connectome columns for Sex_F: {top_conn_cols[:5]}... (showing first 5)")

# Extract these top features
X_conn_top20 = X_conn[top_conn_cols].values
X_test_conn_top20 = X_test_conn[top_conn_cols].values

# 4. For ADHD prediction, focus on metadata features
print("\n🔍 Preparing specialized features for ADHD prediction...")
X_meta = X.drop(columns=connectome_cols).copy()
X_test_meta = X_test.drop(columns=connectome_cols).copy()

# Identify important metadata features for ADHD using a quick GBM
print("Finding top metadata features for ADHD prediction...")
quick_gbm = GradientBoostingClassifier(n_estimators=100, random_state=42)
quick_gbm.fit(X_meta, y['ADHD_Outcome'])
feature_importance = quick_gbm.feature_importances_
top_20_meta_indices = np.argsort(feature_importance)[-20:]
top_meta_cols = [X_meta.columns[i] for i in top_20_meta_indices]
print(f"Top metadata features for ADHD: {top_meta_cols[:5]}... (showing first 5)")

# Extract these top features
X_meta_top20 = X_meta[top_meta_cols].values
X_test_meta_top20 = X_test_meta[top_meta_cols].values

# 5. Prepare final feature sets for each target
# For Sex_F: Combine connectome features and top metadata
X_sex = np.hstack([train_connectome_features, X_conn_top20, X_meta_top20])
X_test_sex = np.hstack([test_connectome_features, X_test_conn_top20, X_test_meta_top20])
print(f"Final Sex_F feature set shape: {X_sex.shape}")

# For ADHD: Combine metadata features and connectome summary features
X_adhd = np.hstack([X_meta_top20, train_connectome_features])
X_test_adhd = np.hstack([X_test_meta_top20, test_connectome_features])
print(f"Final ADHD feature set shape: {X_adhd.shape}")

# Scale features
scaler_sex = RobustScaler()
X_sex_scaled = scaler_sex.fit_transform(X_sex)
X_test_sex_scaled = scaler_sex.transform(X_test_sex)

scaler_adhd = RobustScaler()
X_adhd_scaled = scaler_adhd.fit_transform(X_adhd)
X_test_adhd_scaled = scaler_adhd.transform(X_test_adhd)

# 📈 Setup fast, optimized models for each target
print("\n🧪 Setting up specialized models...")

# For Sex_F prediction (primarily connectome-based)
sex_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.02,
    depth=7,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=0,
    task_type='GPU',  # Try GPU if available
    thread_count=-1,  # Use all cores
    boosting_type='Plain',
    grow_policy='Lossguide',
    bootstrap_type='Bernoulli',
    subsample=0.8
)

# For ADHD prediction (focus on metadata + connectome features)
adhd_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=63,
    max_depth=7,
    class_weight='balanced',
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    reg_alpha=0.1,
    reg_lambda=0.1,
    n_jobs=-1,  # Use all cores
    random_state=42,
    verbose=-1  # Set verbosity in the model initialization instead
)

# Train Sex_F model
print("\n🧪 Training Sex_F model...")
sex_model.fit(
    X_sex_scaled, 
    y['Sex_F'].values,
    eval_set=[(X_sex_scaled, y['Sex_F'].values)],
    verbose=False
)

# Train ADHD model
print("\n🧪 Training ADHD model...")
adhd_model.fit(
    X_adhd_scaled, 
    y['ADHD_Outcome'].values,
    eval_set=[(X_adhd_scaled, y['ADHD_Outcome'].values)]
)

# Generate predictions with probability calibration
sex_probs = sex_model.predict_proba(X_test_sex_scaled)[:, 1]
adhd_probs = adhd_model.predict_proba(X_test_adhd_scaled)[:, 1]

# Print distribution of probability scores
print("\n📊 Probability Score Distribution:")
print(f"Sex_F probabilities - min: {sex_probs.min():.3f}, max: {sex_probs.max():.3f}, mean: {sex_probs.mean():.3f}")
for p in [10, 20, 30, 40, 50, 60, 70, 80, 90]:
    print(f"Sex_F {p}th percentile: {np.percentile(sex_probs, p):.4f}")

# NEW: FORCE EXACT NUMBER OF SEX_F PREDICTIONS (115)
print("\n🛠️ Forcing exactly 115 Sex_F predictions based on probability ranking...")
# Sort probabilities and find the threshold that gives us exactly 115 predictions
sex_threshold_to_force_115 = np.sort(sex_probs)[-115]
print(f"Using dynamic threshold {sex_threshold_to_force_115:.4f} to get exactly 115 Sex_F predictions")

# Apply thresholds
sex_preds = (sex_probs >= sex_threshold_to_force_115).astype(int)
adhd_threshold = 0.44  # Keep the same for ADHD
adhd_preds = (adhd_probs > adhd_threshold).astype(int)

# Additional post-processing specifically for joint cases
# High-value female ADHD combinations (get 2x weight in scoring)
print("\n🔍 Post-processing to optimize female ADHD combinations...")
# Find the most promising female+ADHD combinations that aren't already captured
potential_female_adhd = (sex_probs > sex_threshold_to_force_115*0.85) & (adhd_probs > 0.5) & (sex_preds == 0)
# Rank them by joint probability
joint_probs = sex_probs * adhd_probs
ranked_potential = np.argsort(joint_probs[potential_female_adhd])

# Select additional high-value combinations (but maintain exactly 115 Sex_F predictions)
# Count current female-ADHD count
current_female_adhd = np.sum((sex_preds == 1) & (adhd_preds == 1))
print(f"Current female ADHD combinations: {current_female_adhd}")

# Optimize for more female-ADHD combinations while keeping total Female count at 115
female_non_adhd = (sex_preds == 1) & (adhd_preds == 0)
# Sort these by lowest sex probability
female_non_adhd_indices = np.where(female_non_adhd)[0]
if len(female_non_adhd_indices) > 0:
    female_non_adhd_sex_probs = sex_probs[female_non_adhd_indices]
    sorted_indices = female_non_adhd_indices[np.argsort(female_non_adhd_sex_probs)]
    
    # Find potential new female-ADHD candidates
    potential_upgrades = (sex_probs > sex_threshold_to_force_115*0.8) & (adhd_probs > 0.6) & (sex_preds == 0)
    potential_indices = np.where(potential_upgrades)[0]
    
    # Sort potential upgrades by joint probability (highest first)
    if len(potential_indices) > 0:
        joint_probs_upgrades = joint_probs[potential_indices]
        potential_indices = potential_indices[np.argsort(-joint_probs_upgrades)]
        
        # Make swaps to maximize Female+ADHD combinations while keeping total Female count at 115
        swaps_made = 0
        max_swaps = min(len(sorted_indices), len(potential_indices), 10)  # Limit to 10 swaps
        
        for i in range(max_swaps):
            # Downgrade a Female+non-ADHD
            sex_preds[sorted_indices[i]] = 0
            # Upgrade a high-potential non-Female+ADHD to Female+ADHD
            sex_preds[potential_indices[i]] = 1
            adhd_preds[potential_indices[i]] = 1
            swaps_made += 1
            
        print(f"Made {swaps_made} swaps to optimize Female+ADHD combinations")

# Report prediction statistics
print("\n📊 Final Prediction Statistics:")
print(f"Sex_F (female) predictions: {np.sum(sex_preds)} out of {len(sex_preds)} ({np.mean(sex_preds)*100:.1f}%)")
print(f"ADHD_Outcome predictions: {np.sum(adhd_preds)} out of {len(adhd_preds)} ({np.mean(adhd_preds)*100:.1f}%)")
print(f"Female ADHD cases (2x weight): {np.sum((sex_preds == 1) & (adhd_preds == 1))}")

# Create submission in the specific format requested: participant_id, ADHD_Outcome, Sex_F
print("\n📝 Creating submission file...")
# Using binary predictions (0/1) as required
submission = pd.DataFrame({
    'participant_id': test_ids,
    'ADHD_Outcome': adhd_preds,  # Using binary predictions (0/1)
    'Sex_F': sex_preds  # Using binary predictions (0/1)
})

# Save submission
submission_path = os.path.join(output_dir, "submission_ultra_optimized_binary_115females.csv")
submission.to_csv(submission_path, index=False)
print(f"\n✅ Submission saved to '{submission_path}'")

# Print execution time
total_time = time.time() - start_time
print(f"\n⏱️ Total execution time: {total_time/60:.2f} minutes")


🔍 Starting WiDS 2025 Ultra-Optimized Classification Pipeline...

📂 Loading TRAIN datasets...

📂 Loading TEST datasets...

🔄 Merging datasets...
Final train dataset shape: (1213, 19930)
Final test dataset shape: (304, 19928)

🔄 Advanced feature engineering...

🔍 Extracting specialized connectome features for Sex_F...
Generated 23 specialized connectome features

🔍 Finding top connectome columns for Sex_F prediction...
Top 20 connectome columns for Sex_F: ['129throw_169thcolumn', '25throw_86thcolumn', '77throw_157thcolumn', '83throw_192thcolumn', '29throw_94thcolumn']... (showing first 5)

🔍 Preparing specialized features for ADHD prediction...
Finding top metadata features for ADHD prediction...
Top metadata features for ADHD: ['Barratt_Barratt_P1_Occ', 'Barratt_Barratt_P2_Edu', 'SDQ_SDQ_Prosocial', 'MRI_Track_Scan_Location', 'EHQ_EHQ_Total']... (showing first 5)
Final Sex_F feature set shape: (1213, 63)
Final ADHD feature set shape: (1213, 43)

🧪 Setting up specialized models...

🧪 Tra

In [2]:
# === Updated V19 Best Model with Full Research-Level Graphs (Fixed SHAP Bug) ===

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score, roc_curve, auc, precision_recall_curve, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV

import shap
import matplotlib.pyplot as plt
import seaborn as sns
from docx import Document
from docx.shared import Inches

import tempfile

# === Paths ===
data_dir = r"C:\Users\sanja\Desktop\WiDS"
train_X_path = os.path.join(data_dir, "CLEANED_DATA", "FINAL_preprocessed_train_v1.csv")
train_y_path = os.path.join(data_dir, "CLEANED_DATA", "FINAL_labels_train_v1.csv")
test_X_path = os.path.join(data_dir, "CLEANED_DATA", "FINAL_preprocessed_test_v1.csv")
test_cat_path = os.path.join(data_dir, "TEST", "TEST_CATEGORICAL.xlsx")

submission_path = os.path.join(data_dir, "submission_final_v19_best_graphs.csv")
report_path = os.path.join(data_dir, "model_report_v19_best_graphs.docx")

# === Load Data ===
X_train = pd.read_csv(train_X_path)
y_df = pd.read_csv(train_y_path)
X_test = pd.read_csv(test_X_path)
test_ids = pd.read_excel(test_cat_path)['participant_id']

y_adhd = y_df['ADHD_Outcome']
y_sex  = y_df['Sex_F']

# === Word Report ===
doc = Document()
doc.add_heading("Model Report - Best V19 (with Graphs)", 0)

# === Plot Helper ===
def plot_and_insert(doc, fig, title):
    with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tmpfile:
        fig.savefig(tmpfile.name, bbox_inches='tight')
        plt.close(fig)
        doc.add_picture(tmpfile.name, width=Inches(5))

# === Model Training Function ===
def run_model(X, y, X_test, name, sex_weight=None):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    test_preds = np.zeros(X_test.shape[0])
    all_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        base_models = [
            ('lgb', LGBMClassifier(n_estimators=300, random_state=42)),
            ('cat', CatBoostClassifier(verbose=0, iterations=300, random_state=42))
        ]

        stack = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(max_iter=500))
        model = CalibratedClassifierCV(stack, method='sigmoid')
        model.fit(X_tr, y_tr)

        val_probs = model.predict_proba(X_val)[:, 1]
        best_score = 0
        best_t = 0.5

        for t in np.arange(0.2, 0.81, 0.01):
            preds = (val_probs > t).astype(int)
            score = f1_score(y_val, preds, average='macro') if name == 'Sex_F' else f1_score(y_val, preds, sample_weight=(2*sex_weight[val_idx] + 1))
            if score > best_score:
                best_score = score
                best_t = t

        final_preds = (val_probs > best_t).astype(int)
        cm = confusion_matrix(y_val, final_preds)
        report = classification_report(y_val, final_preds)

        doc.add_heading(f"{name} - Fold {fold+1}", level=2)
        doc.add_paragraph(f"Best Threshold: {best_t:.2f}")
        doc.add_paragraph(f"Optimized Score: {best_score:.4f}")
        doc.add_paragraph(report)

        # Confusion Matrix
        fig, ax = plt.subplots(figsize=(4, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f"{name} Fold {fold+1} Confusion Matrix")
        plot_and_insert(doc, fig, "Confusion Matrix")

        # ROC Curve
        fpr, tpr, _ = roc_curve(y_val, val_probs)
        roc_auc = auc(fpr, tpr)
        fig, ax = plt.subplots()
        ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        ax.plot([0, 1], [0, 1], linestyle='--')
        ax.set_title(f"{name} Fold {fold+1} ROC Curve")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend()
        plot_and_insert(doc, fig, "ROC")

        # Precision-Recall Curve
        prec, rec, _ = precision_recall_curve(y_val, val_probs)
        fig, ax = plt.subplots()
        ax.plot(rec, prec)
        ax.set_title(f"{name} Fold {fold+1} Precision-Recall Curve")
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        plot_and_insert(doc, fig, "PR Curve")

        # SHAP Summary (only on 1st fold and only if using LGBM directly)
        if fold == 0:
            raw_lgb_model = base_models[0][1]  # LGBMClassifier
            raw_lgb_model.fit(X_tr, y_tr)
            explainer = shap.Explainer(raw_lgb_model, X_tr)
            shap_values = explainer(X_val)
            fig = shap.plots.beeswarm(shap_values, show=False)
            plot_and_insert(doc, plt.gcf(), "SHAP Summary")

        test_preds += (model.predict_proba(X_test)[:, 1] > best_t).astype(int)
        all_scores.append(best_score)

    avg_score = np.mean(all_scores)
    doc.add_heading(f"{name} - Average Optimized Score", level=2)
    doc.add_paragraph(f"{avg_score:.4f}")
    return (test_preds / skf.n_splits > 0.5).astype(int)

# === Run Models ===
pred_adhd = run_model(X_train, y_adhd, X_test, 'ADHD_Outcome', sex_weight=y_sex)
pred_sex = run_model(X_train, y_sex, X_test, 'Sex_F')

# === Final Submission ===
final_df = pd.DataFrame({
    'participant_id': test_ids,
    'ADHD_Outcome': pred_adhd,
    'Sex_F': pred_sex
})
final_df.to_csv(submission_path, index=False)

# === Save Report ===
doc.save(report_path)
print(f"✅ Submission saved to: {submission_path}")
print(f"✅ Report saved to: {report_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sanja\\Desktop\\WiDS\\CLEANED_DATA\\FINAL_preprocessed_train_v1.csv'